In [ ]:
from pathlib import Path
from data_wrangler import DataWrangler, LocalCsvDataSource

In [ ]:
# Configure logging
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


In [ ]:
DOWNLOAD_PATH = Path("../../data/land_registry_data/")
assert DOWNLOAD_PATH.exists(), f"Download path {DOWNLOAD_PATH} does not exist. Please download the data and place it in this directory."

In [ ]:
ONELAKE_PATH = Path("../../data/onelake")
assert ONELAKE_PATH.exists(), f"OneLake path {ONELAKE_PATH} does not exist. Please set up OneLake and place the data in this directory."

In [ ]:
data_source = LocalCsvDataSource(
    data_folder=str(DOWNLOAD_PATH),
    column_names=DataWrangler.COLUMN_NAMES,
    output_root=str(ONELAKE_PATH)
)

data_wrangler = DataWrangler(data_source)

house_price_summary = data_wrangler.process_to_silver()

In [ ]:
house_price_summary = data_wrangler.project_to_gold()

## Consume Data From Gold Layer

In [ ]:
import polars as pl

In [ ]:
prices = pl.read_delta(ONELAKE_PATH / "gold" / "fact_price_paid")
dates = pl.read_delta(ONELAKE_PATH / "gold" / "dim_date")

In [ ]:
house_price_summary = (
    prices
    .join(dates, left_on="date_of_transfer", right_on="date")
    .group_by(["year", "property_type"])
    .agg(pl.median("price").alias("median_price")).sort(["year", "property_type"])
)

In [ ]:
import plotly.express as px
fig = px.line(
    house_price_summary,
    x="year",
    y="median_price",
    color="property_type",
    markers=True,
    title="Median House Price by Year and Property Type",
    labels={"year": "Year", "median_price": "Median Price (£)", "property_type": "Property Type"},
)
fig.show()